## Gold — Hourly Coverage

Calculates for every daily hours and day type how many active trips and stops per city from silver tables.

### Source
- `gtfs_silver.trips` — trips linked to routes and services
- `gtfs_silver.stop_times` — stops served per trip
- `gtfs_silver.calendar_dates` — active service dates

### Output
`gtfs_gold.hourly_coverage`

| Column | Description |
|--------|-------------|
| city | City name |
| hour_of_day | distinct hour of day |
| day_type | type of day |
| active_trips | trips active for hour and day |
| active_routes | routes active for hour and day |
| stops_served | stop served for hour and day |

### CTE Logic
- **trip_day_type** — joins `trips` → `calendar_dates` to count distinct trips per day type
- **stop_times_with_hours** — calculates the hour of the day from departure_secs


In [0]:
CREATE OR REPLACE TABLE gtfs_gold.hourly_coverage AS
WITH trip_day_type AS (
    SELECT DISTINCT 
        t.route_id,
        t.city,
        t.trip_id,
        CASE WHEN DAYOFWEEK(cd.date) BETWEEN 2 AND 6 THEN 'Weekday'
             WHEN DAYOFWEEK(cd.date) = 7 THEN 'Saturday'
             ELSE 'Sunday'
        END AS day_type
    FROM gtfs_silver.trips t
    JOIN gtfs_silver.calendar_dates cd ON t.service_id = cd.service_id AND t.city=cd.city
),
stop_times_with_hour AS (
    SELECT 
        *,
        LEAST(FLOOR(departure_secs / 3600), 23) AS hour_of_day
    FROM gtfs_silver.stop_times
)
SELECT 
    tdt.city,
    tdt.day_type,
    sth.hour_of_day,
    COUNT(DISTINCT tdt.trip_id)  AS active_trips,
    COUNT(DISTINCT sth.stop_id)  AS active_stops,
    COUNT(DISTINCT tdt.route_id) AS active_routes 
FROM trip_day_type tdt
JOIN stop_times_with_hour sth ON tdt.city = sth.city AND tdt.trip_id = sth.trip_id
GROUP BY tdt.city, tdt.day_type, sth.hour_of_day;

In [0]:
SELECT city, day_type, hour_of_day, active_trips, active_stops
FROM gtfs_gold.hourly_coverage
WHERE day_type = 'Weekday'
ORDER BY hour_of_day